In [19]:
! python -m pip install numpy scipy matplotlib
import scipy as scp
import numpy as np
import matplotlib.pyplot as plt

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


   ---------------------------------------- 0.0/12.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/12.6 MB ? eta -:--:--
   -- ------------------------------------- 0.8/12.6 MB 2.9 MB/s eta 0:00:05
   ---- ----------------------------------- 1.3/12.6 MB 2.7 MB/s eta 0:00:05
   ----- ---------------------------------- 1.8/12.6 MB 2.6 MB/s eta 0:00:05
   ------ --------------------------------- 2.1/12.6 MB 2.4 MB/s eta 0:00:05
   -------- ------------------------------- 2.6/12.6 MB 2.3 MB/s eta 0:00:05
   --------- ------------------------------ 3.1/12.6 MB 2.4 MB/s eta 0:00:04
   ----------- ---------------------------- 3.7/12.6 MB 2.3 MB/s eta 0:00:04
   ------------- -------------------------- 4.2/12.6 MB 2.4 MB/s eta 0:00:04
   ---------------- ----------------------- 5.2/12.6 MB 2.6 MB/s eta 0:00:03
   ------------------ --------------------- 5.8/12.6 MB 2.6 MB/s eta 0:00:03
   ------------------- -------------------- 6.0/12.6 MB 2.5 MB/s eta 0:00:03
   ----------

In [ ]:
csi_largo = np.random.randn(100, 8) + 1j * np.random.randn(100, 8)

In [ ]:
csi_ex = np.array([
    # Paquete 1 (t = 0)
    [13.2 + 7.2j,  5.4 + 14.0j, -6.2 + 13.6j, -14.1 + 5.0j,
    -13.4 - 6.6j, -4.6 - 14.3j,  7.0 - 13.3j,  14.4 - 4.2j],
    
    # Paquete 2 (t = 1)
    [11.7 + 9.3j,  3.2 + 14.6j, -8.3 + 12.5j, -14.7 + 3.0j,
    -11.8 - 9.2j, -2.5 - 14.8j,  8.8 - 12.1j,  14.8 - 2.3j],
    
    # Paquete 3 (t = 2)
    [ 9.9 + 11.2j,  0.9 + 15.0j, -10.2 + 11.0j, -14.9 + 0.9j,
    -9.9 - 11.2j, -0.3 - 15.0j,  10.4 - 10.8j,  14.9 - 0.3j]
])

fases = []


In [ ]:
for subportadora in csi_largo:
    fases_z =[]
    for z in subportadora:
        I = z.real  # parte real
        Q = z.imag # parte imaginaria 
        fase = np.arctan2(Q, I) 
        fases_z.append(fase)
        
    fases.append(fases_z)
fases_brutas = np.array(fases)
fases_unwrapped = np.unwrap(fases_brutas, axis=1) 
print(fases_unwrapped)

In [ ]:
fs = 100 #asumo que la frecuencia d elo que me mande hardware será 100 paquetes x seg
nyquist = fs / 2 # tiene que ser la mitad de lo que recibe pq si no tosquea x alguna razon. se llama limite de nyquist
frecuencia_baja, frecuencia_alta = 0.1, 0.5 # 0.1 son 6rpm y 0.5 30rpm
low = frecuencia_baja / nyquist
high = frecuencia_alta / nyquist
b, a = scp.signal.butter(N=2, Wn=[low, high], btype='bandpass') #plantilla del filtro
fases_filtradas = scp.signal.filtfilt(b, a, fases_unwrapped, axis=0) #aplico el filtro a mi fase
print(fases_filtradas)


In [ ]:
señal = fases_filtradas[:, 0] #todas las filas, solo la columna 0
fft_valores = np.fft.fft(señal) #aplico fft que descompone onda en senos y cosenos. pasa de time domain a frequency domain
magnitudes = np.abs(fft_valores) #saca el módulo
frecuencias = np.fft.fftfreq(len(señal), d=1/fs) #da un valor que se corresponde con cada uno de los que obtengo del fft
mitad = len(señal) // 2#fft devuelve espejado, positivo y negativo, y yo necesito solo uno asi que divido x 2
frecuencias_pos = frecuencias[:mitad]
magnitudes_pos = magnitudes[:mitad] #agarro de 0 a la mitad
indice_pico = np.argmax(magnitudes_pos)
rpm = frecuencias_pos[indice_pico] * 60
print(rpm)

0.0


In [29]:
# 1. Crear un CSI Sintético que SÍ contiene respiración a 15 RPM (0.25 Hz)
t = np.linspace(0, 5, 500)  # 5 segundos a 100 Hz (500 paquetes)
fs = 100

# Onda senoidal de 0.25 Hz (15 RPM) + Ruido
frecuencia_respiracion = 0.25 
patron_respiratorio = np.sin(2 * np.pi * frecuencia_respiracion * t)
ruido = np.random.normal(0, 0.2, 500)

# Generamos CSI complejo donde la FASE oscila con la respiración
fase_simulada = patron_respiratorio + ruido
csi_largo = np.exp(1j * fase_simulada).reshape(500, 1) # 500 paquetes, 1 subportadora

# 2. Extracción de fase y Unwrap
fases_brutas = np.angle(csi_largo)
fases_unwrapped = np.unwrap(fases_brutas, axis=0)

# 3. Filtro Butterworth
nyquist = fs / 2
low, high = 0.1 / nyquist, 0.5 / nyquist
b, a = scp.signal.butter(N=2, Wn=[low, high], btype='bandpass')
fases_filtradas = scp.signal.filtfilt(b, a, fases_unwrapped, axis=0)

# 4. FFT con Zero-Padding (n_fft=10000 para dar resolución fina)
señal = fases_filtradas[:, 0]
n_fft = 10000 

fft_valores = np.fft.fft(señal, n=n_fft)
magnitudes = np.abs(fft_valores)
frecuencias = np.fft.fftfreq(n_fft, d=1/fs)

# 5. Tomar mitad positiva
mitad = n_fft // 2
frecuencias_pos = frecuencias[:mitad]
magnitudes_pos = magnitudes[:mitad]

# 6. Convertir TODO el vector a RPM
rpm_pos = frecuencias_pos * 60

# 7. Aplicar la máscara humana (6 a 30 RPM) sobre el VECTOR
rpmshumanas = (rpm_pos >= 6) & (rpm_pos <= 30)
rpm_validas = rpm_pos[rpmshumanas]
magnitudes_validas = magnitudes_pos[rpmshumanas]

# 8. Buscar el pico en el rango válido
if len(magnitudes_validas) > 0:
    indice_pico = np.argmax(magnitudes_validas)
    rpm_detectadas = rpm_validas[indice_pico]
    print(f"RPM Detectadas: {rpm_detectadas:.1f} RPM")
else:
    print("No hay suficientes datos para evaluar el rango de respiración.")

RPM Detectadas: 11.4 RPM
